# 05 — Tabular records

**Workload:** Structured arrays, generated CSV input, `loadtxt`/`genfromtxt`, sorting, and record joins.

This notebook is executed against the RNP engine. Every output below is
stored in the notebook and visible when rendered on GitHub.

In [1]:
from pathlib import Path
import importlib.util
import sys

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "shim" / "rnp_numpy").is_dir()
)
for path in (PROJECT_ROOT / "harness" / "_redirect", PROJECT_ROOT / "shim"):
    sys.path.insert(0, str(path))

# IPython may preload the oracle NumPy, so clear that namespace before
# executing the exact redirect hook used by examples/run_all.py.
for module_name in list(sys.modules):
    if module_name == "numpy" or module_name.startswith("numpy."):
        del sys.modules[module_name]
redirect_path = PROJECT_ROOT / "harness" / "_redirect" / "sitecustomize.py"
redirect_spec = importlib.util.spec_from_file_location("_rnp_notebook_redirect", redirect_path)
redirect = importlib.util.module_from_spec(redirect_spec)
redirect_spec.loader.exec_module(redirect)
import numpy as np

probe = np.array(0)
print("numpy version:", np.__version__)
print(f"RNP engine active: {np.__name__} ({type(probe).__module__}.{type(probe).__name__})")
assert np.__name__ == "rnp_numpy"

numpy version: 2.5.2
RNP engine active: rnp_numpy (_rnp.ndarray)


## Generate and parse CSV data

Create both numeric and named-record input entirely in memory.

In [2]:
import io

table = np.array([[101.0, 8.5], [102.0, 9.25], [103.0, 7.75]])
numeric_csv = io.StringIO()
np.savetxt(numeric_csv, table, delimiter=",", fmt=["%.0f", "%.2f"])
numeric_csv.seek(0)
loaded_numeric = np.loadtxt(numeric_csv, delimiter=",")

rows = ["id,name,score", "103,Linus,7.75", "101,Ada,8.50", "102,Grace,9.25"]
record_csv = io.StringIO("\n".join(rows))
records = np.genfromtxt(record_csv, delimiter=",", names=True, dtype=None, encoding=None)
print("loaded numeric rows:\n", loaded_numeric)
print("records:", records)

loaded numeric rows:
 [[101.     8.5 ]
 [102.     9.25]
 [103.     7.75]]
records: [(103, 'Linus', 7.75) (101, 'Ada', 8.5 ) (102, 'Grace', 9.25)]


## Sort and join records

RNP does not yet implement `sort(order=)` or a fully faithful
`recfunctions.join_by`. As documented in
[KNOWN_GAPS.md](../KNOWN_GAPS.md), this uses equivalent public
`argsort` and `searchsorted` formulations.

In [3]:
sorted_records = records[np.argsort(records["score"])[::-1]]
bonuses = np.array(
    [(101, 1.5), (102, 2.0), (103, 1.0)],
    dtype=[("id", "i8"), ("bonus", "f8")],
)
left = records[np.argsort(records["id"])]
right = bonuses[np.argsort(bonuses["id"])]
right_rows = np.searchsorted(right["id"], left["id"])
matched = right["id"][right_rows] == left["id"]
joined_ids = left["id"][matched]
joined_bonus = right["bonus"][right_rows[matched]]
print("ranked names:", sorted_records["name"])
print("joined IDs:", joined_ids)
print("joined bonuses:", joined_bonus)

ranked names: ['Grace' 'Ada' 'Linus']
joined IDs: [101 102 103]
joined bonuses: [1.5 2.  1. ]


## Verify the result

In [4]:
assert np.allclose(loaded_numeric, [[101.0, 8.5], [102.0, 9.25], [103.0, 7.75]], rtol=0.0, atol=0.0)
assert np.array_equal(sorted_records["id"], [102, 101, 103])
assert np.array_equal(sorted_records["name"], ["Grace", "Ada", "Linus"])
assert np.array_equal(joined_ids, [101, 102, 103])
print("PASS — all tabular-record assertions passed.")

PASS — all tabular-record assertions passed.
